# 02 · Stimuli and Validation

Audit four practice and transfer pairs over the complete input domain. Upload the project ZIP when prompted.

In [ ]:
import io
import json
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

root = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd() / "check-before-you-run")
             if (p / "src" / "study.py").is_file()), None)
if root is None:
    from google.colab import files
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Upload the original project ZIP.")
    with ZipFile(io.BytesIO(next(iter(uploads.values())))) as archive:
        paths = [PurePosixPath(name) for name in archive.namelist()]
        if not paths or any(path.is_absolute() or ".." in path.parts for path in paths):
            raise ValueError("Invalid project archive.")
        archive.extractall(Path.cwd())
    root = Path.cwd() / "check-before-you-run"
    if not (root / "src" / "study.py").is_file():
        raise ValueError("Use the original check-before-you-run.zip.")
root = root.resolve()
sys.path.insert(0, str(root))
from src import study

print("Project ready")

In [ ]:
for task_id in study.TASKS:
    print(f"\n{task_id}")
    for phase, is_transfer in (("practice", False), ("transfer", True)):
        task = study.fixture(task_id, is_transfer)
        failures = [x for x in study.INPUTS if task["expected"][x] != task["actual"][x]]
        assert failures and len(failures) < len(study.INPUTS)
        if not is_transfer:
            assert set(task["choices"]) & set(failures)
            assert set(task["choices"]) - set(failures)
        print(f"{phase}: {task['name']}")
        print(task["spec"])
        print(task["code"])
        print(f"Failure-inducing inputs: {failures}")